# DR – Druhá semestrální práce

## Zadání

1. Získejte datovou sadu průměrných denních teplot v Plzni za období od 1. ledna 1970 do 12. prosince 2025.

2. Rozdělte datovou sadu podle jednotlivých kalendářních roků.

3. Vykreslete graf průměrných ročních teplot. V samostatném grafu zobrazte průběh denních teplot v jednotlivých letech.

4. Pomocí **nepárového t-testu** zjistěte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

5. Pomocí **párového t-testu** zjistěte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

6. Vykreslete histogram odchylek a posuďte, zda jsou splněny předpoklady pro použití t-testu.

7. Použijte vhodný **párový neparametrický** test a vyhodnoťte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

8. Porovnejte výsledky nepárového a párového t-testu. Vyhodnoťte, zda se statisticky významně odlišuje více, nebo méně roků při použití nepárového či párového testu. Stejně tak porovnejte výsledky párového t-testu a zvoleného párového neparametrického testu. Pokuste se vysvětlit případné rozdíly ve výsledcích.

9. Vyhodnoťte trend průměrných ročních teplot a otestujte hypotézu, že tento trend je statisticky významně odlišný od nuly.

## Dodatečné informace z emailu
Zadání opět vypacujte ve formě ipython notebooku a zašlete mi ho.

Jako úkol nad rámec zadání si můžete širokou paletu možnosti od srovnání s jinými místy planety až po analýzu, které dny v roce se více hodí pro porovnávání teplot. V zadání je také schválně ponechána jedna (a neúmyslně možná více) nepřesnost/nevhodnost. Její ohalení a náprava také může být úkolem nad rámec zadání.

Varování: pozor na příliš časté používání API zvolené služby. Nezapomínejte vyčkat několik sekund před opakováním dotazu.

## Vypracování

### 1. Získání datavé sady

Souřadnice Plzně jsou 49° 44′ 51″ N, 13° 22′ 39″ E (zdroj: https://en.wikipedia.org/wiki/Plze%C5%88).

Následuje definice parametrů datasetu:

In [ ]:
# --------------------------------------------------
# Configuration
# --------------------------------------------------

location = {
    "name": "Pilsen",
    "continent": "Europe",
    "timezone": "Europe/Prague",
    "latitude": (49, 44, 51),
    "longitude": (13, 22, 39),
}

start_date = "1970-01-01"
# end_date = "2025-12-31"
end_date = "2025-12-12" # zadani rika 12. prosince, ale tady bylo napsano 31. prosince
# Preklep, nebo kontrola toho ze jsem cetl zadani?
# nebo kontrola toho ze to byla hozeny cely do LLMka?
# who knows

# --------------------------------------------------
# Derived parameters
# --------------------------------------------------

for key in ("latitude", "longitude"):
    degrees, minutes, seconds = location[key]
    location[key] = degrees + minutes / 60 + seconds / 3600

# --------------------------------------------------
# Output
# --------------------------------------------------

print(f"Location: {location['name']}")
print(f"Continent: {location['continent']}")
print(f"Timezone: {location['timezone']}")
print(f"Latitude: {location['latitude']:.3f}")
print(f"Longitude: {location['longitude']:.3f}")
print(f"Start date: {start_date}")
print(f"End date: {end_date}")

Následuje výpočet, kolik bodů pozorování bychom měli dostat.

In [ ]:
import datetime

date_start = datetime.date.fromisoformat(start_date)
date_end = datetime.date.fromisoformat(end_date)

n_days_expected = (date_end - date_start).days + 1

print(f"Start date: {date_start}")
print(f"End date: {date_end}")
print(f"Expected number of observations: {n_days_expected:,}")

K získání dat byla použita otavřená data s url adresou v následujícím kódu. Tento kód provádí záskání dat a kontrolu jejich správnosti.

In [ ]:
import requests
import pandas as pd
import json

# --------------------------------------------------
# Open-Meteo API
# --------------------------------------------------

url = "https://archive-api.open-meteo.com/v1/archive"

daily_variables = [
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
]

params = {
    "latitude": location["latitude"],
    "longitude": location["longitude"],
    "start_date": start_date,
    "end_date": end_date,
    "daily": daily_variables,
    "timezone": location["timezone"],
    "models": "era5",
}

# --------------------------------------------------
# Download data
# --------------------------------------------------

# Pro prevenci příliš častého volání api jsem po prvním spuštění data uložil do souboru data.json
# with requests.Session() as session:
#     response = session.get(url, params=params, timeout=(10, 120))
#     response.raise_for_status()
#     result = response.json()

# json_str = json.dumps(result, indent=2)
# with open("data_Pilsen.json", "w") as f:
#     f.write(json_str)


with open("data_Pilsen.json") as f:
    result = json.load(f)



# --------------------------------------------------
# Extract data
# --------------------------------------------------

if "daily" not in result:
    raise ValueError("Daily data are missing from the API response.")

daily = result["daily"]

df = pd.DataFrame({
    "date": pd.to_datetime(daily["time"]),
    "temperature_mean": daily["temperature_2m_mean"],
    "temperature_min": daily["temperature_2m_min"],
    "temperature_max": daily["temperature_2m_max"],
})

# --------------------------------------------------
# Data validation
# --------------------------------------------------

print(f"Obtained observations: {len(df):,}")
print(f"Expected observations: {n_days_expected:,}")

if len(df) != n_days_expected:
    print("WARNING: Unexpected number of observations!")

if df["date"].duplicated().any():
    raise ValueError("Duplicate dates detected.")

if not df["date"].is_monotonic_increasing:
    raise ValueError("Dates are not in chronological order.")

missing = df.isna().sum()

if missing.any():
    print("\nMissing values:")
    print(missing[missing > 0])

# --------------------------------------------------
# API metadata
# --------------------------------------------------

print("\nTemperature units:")
print(result.get("daily_units", {}))

print(f"\nTimezone: {result.get('timezone')}")
print(f"Elevation: {result.get('elevation')} m")

df.head()

Následující kód vykreslí všechny body datové sady.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --------------------------------------------------
# Plot daily temperature
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    df["date"],
    df["temperature_mean"],
    linewidth=0.4,
    alpha=0.7,
)

ax.set_title(f"Daily Mean Temperature in {location['name']}")
ax.set_ylabel("Temperature [°C]")
ax.set_xlabel("Date")

# Date formatting
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### 2. Rozdělení datové sady

In [ ]:
# --------------------------------------------------
# Split data into annual DataFrames
# --------------------------------------------------

df["year"] = df["date"].dt.year

df_annual = {
    year: group.drop(columns="year").reset_index(drop=True)
    for year, group in df.groupby("year")
}

print(f"Number of years: {len(df_annual)}")
print(f"Available years: {list(df_annual.keys())}")

df_annual[2025]

### 3. Vykreslení průběhů teplot

Následují kód vykresluje vývoj průměrné roční teploty.

In [ ]:
# --------------------------------------------------
# Annual temperature statistics
# --------------------------------------------------

df_annual_stats = pd.DataFrame({
    year: {
        "mean": data["temperature_mean"].mean(),
    }
    for year, data in df_annual.items()
}).T
df_annual_stats.index.name = "year"

# --------------------------------------------------
# Plot annual mean temperature ± standard deviation
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 5))
years = df_annual_stats.index
means = df_annual_stats["mean"]
ax.plot(years, means, color="blue", linewidth=2.0)
ax.set_title(f"Annual Mean Temperature in {location['name']}")
ax.set_xlabel("Year")
ax.set_ylabel("Temperature [°C]")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Následující kód vykresluje průběh teplot během roku a průměrný rok.

**Pozor rok 2025 není úplný!**
- dojde k mírnému zkreslení výsledků
- (viz úkoly nad rámec zadání)

In [ ]:
# --------------------------------------------------
# Average annual temperature profile
# --------------------------------------------------

df["day_of_year"] = df["date"].dt.day_of_year

df_average_year = (
    df.groupby("day_of_year")["temperature_mean"]
    .agg(["mean", "std"])
    .reset_index()
)

df_average_year.head()

# --------------------------------------------------
# Plot temperature profiles for all years
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 6))

for year, data in df_annual.items():
    ax.plot(
        data["date"].dt.day_of_year,
        data["temperature_mean"],
        color="blue",
        linewidth=0.5,
        alpha=0.15,
    )

# Average annual profile
ax.plot(
    df_average_year["day_of_year"],
    df_average_year["mean"],
    linewidth=2.0,
    color="blue",
    label="Average year",
)

ax.set_title(f"Annual Temperature Profiles in {location['name']}")
ax.set_xlabel("Day of year")
ax.set_ylabel("Temperature [°C]")

ax.set_xlim(1, 366)
ax.set_xticks(range(1, 367, 30))

ax.grid(alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

# Zadání (abych nemusel scrollovat nahoru a dolů)
4. Pomocí **nepárového t-testu** zjistěte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

5. Pomocí **párového t-testu** zjistěte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

6. Vykreslete histogram odchylek a posuďte, zda jsou splněny předpoklady pro použití t-testu.

7. Použijte vhodný **párový neparametrický** test a vyhodnoťte, které roky se statisticky významně liší od celkového průměru ročních teplot. Určete, které roky jsou statisticky významně chladnější a které teplejší než tento průměr. Použijte hladinu významnosti 0.01.

8. Porovnejte výsledky nepárového a párového t-testu. Vyhodnoťte, zda se statisticky významně odlišuje více, nebo méně roků při použití nepárového či párového testu. Stejně tak porovnejte výsledky párového t-testu a zvoleného párového neparametrického testu. Pokuste se vysvětlit případné rozdíly ve výsledcích.

9. Vyhodnoťte trend průměrných ročních teplot a otestujte hypotézu, že tento trend je statisticky významně odlišný od nuly.

### 4. Použití nepárových testů


**Pozor rok 2025 není úplný!**
- dojde k mírnému zkreslení výsledků
- (viz úkoly nad rámec zadání)

Obecná formulace hypotéz:

* H0: odchylka od průměru je způsobena náhodnými poruchami.
* H1: odchylka od průměru je způsobena tím, že datová sada má jiný/nižší/vyšší průměr.

Konkrétní formulace při použití nepárového t-testu:

\begin{align}
H_0:& \mu_\text{rok} = \mu_\text{avg},\\
H_1 (\text{rok je jiný}):& \mu_\text{rok} \ne \mu_\text{avg},\\
H_1 (\text{rok je chladnější}):& \mu_\text{rok} < \mu_\text{avg},,\\
H_1(\text{rok je teplejší}):& \mu_\text{rok} > \mu_\text{avg},\\
\end{align}

In [ ]:
tests_results = {}

In [ ]:
from scipy.stats import ttest_ind

test_name = "t-test"
test_type = "independent"

tests_results[test_name] = {
        test_type: {
            "two-sided": {},
            "less": {},
            "greater": {},
        },
}

average_year = df_average_year["mean"]

# df_annual obsahuje data temp_mean, temp_min, temp_max a dny a je to groupnute podle roku
for year, data in df_annual.items():

    actual_year = data["temperature_mean"] # vsechny data v roce

    p_value_two_sided = ttest_ind(actual_year,
                                   average_year,
                                   equal_var=False,
                                   alternative="two-sided").pvalue
    p_value_less = ttest_ind(actual_year,
                             average_year,
                             equal_var=False,
                             alternative="less").pvalue
    p_value_greater = ttest_ind(actual_year,
                                 average_year,
                                 equal_var=False,
                                 alternative="greater").pvalue  

    tests_results[test_name][test_type]["two-sided"][year] = p_value_two_sided
    tests_results[test_name][test_type]["less"][year] = p_value_less
    tests_results[test_name][test_type]["greater"][year] = p_value_greater

Vykreslení průběhu p-hodnot pro jednotlivé roky.

In [ ]:
def plot_test_results(results, alpha=0.01, title="Grafické zobrazení p-hodnot"):

    years_list = list(results["two-sided"].keys())
    p_two = list(results["two-sided"].values())
    p_less = list(results["less"].values())
    p_greater = list(results["greater"].values())

    # Výpis statisticky významných let na hladině alpha = 0.01
    sig_different = [y for y, p in results["two-sided"].items() if p < alpha]
    sig_colder = [y for y, p in results["less"].items() if p < alpha]
    sig_warmer = [y for y, p in results["greater"].items() if p < alpha]

    print(f"Hladina významnosti alpha = {alpha}")
    print(f"Významně odlišné roky (two-sided) [{len(sig_different)}]: {sig_different}")
    print(f"Významně chladnější roky (less)   [{len(sig_colder)}]: {sig_colder}")
    print(f"Významně teplejší roky (greater)  [{len(sig_warmer)}]: {sig_warmer}")

    # Vykreslení průběhu p-hodnot
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.set_title(title);
    ax.plot(years_list, p_two, marker="o", markersize=4, linestyle="-", alpha=0.8, label="two-sided (jiný)")
    ax.plot(years_list, p_less, marker="v", markersize=4, linestyle="--", alpha=0.7, color="tab:blue", label="less (chladnější)")
    ax.plot(years_list, p_greater, marker="^", markersize=4, linestyle="--", alpha=0.7, color="tab:red", label="greater (teplejší)")

    ax.axhline(y=alpha, color="black", linestyle=":", linewidth=1.5, label=f"Hladina významnosti $\\alpha = {alpha}$")

    ax.set_xlabel("Rok")
    ax.set_ylabel("p-hodnota")
    ax.set_ylim(-0.02, 1.05)
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    plt.show()


    # ax.set_title("")
results = tests_results["t-test"]["independent"]
plot_test_results(results, title="Nepárový Welchův t-test: průběh p-hodnot v jednotlivých letech")

### Rok 2025 (bude relevatntní později)

In [ ]:
print(tests_results["t-test"]["independent"]["two-sided"][2025])

print(tests_results["t-test"]["independent"]["greater"][2025])

print(tests_results["t-test"]["independent"]["less"][2025])

In [ ]:
print(tests_results["t-test"]["independent"]["two-sided"][2022])

print(tests_results["t-test"]["independent"]["greater"][2022])

Zajímavá skutečnost: 
- pro roky zamítáme $H_0$ 2022, 2023 i 2024 ve prospěch $H_1(\text{rok je teplejší}): \mu_\text{rok} > \mu_\text{avg}$
- pro roky 2023, 2024 zamítáme $H_0$ ve prospěch $H_1 (\text{rok je jiný}): \mu_\text{rok} \ne \mu_\text{avg}$, ale to se nestane pro rok 2022
- proč se tomu tak děje?
- oboustranný test dělí hladinu významnosti na dvě poloviny 
- p-hodnota u oboustraného testu je poté dvojnásobkem p-hodnoty jednostranného testu 
- u roku 2022 vyšla jednostranná p-hodnota mezi $0.005$ a $0.01$, což stačí na zamítnutí $H_0$, ale po vynásobení dvěma přeskočila hranici $\alpha$ pro oboustranný test


### 5. Použití párových testů

Obecná formulace hypotéz:

* H0: odchylka od průměru je způsobena náhodnými poruchami.
* H1: odchylka od průměru je způsobena tím, že datová sada má jiný/nižší/vyšší průměr.

Konkrétní formulace při použití párového t-testu:

\begin{align}
H_0:& \mu_\text{rok} = \mu_\text{avg},\\
H_1 (\text{rok je jiný}):& \mu_\text{rok} \ne \mu_\text{avg},\\
H_1 (\text{rok je chladnější}):& \mu_\text{rok} < \mu_\text{avg},,\\
H_1(\text{rok je teplejší}):& \mu_\text{rok} > \mu_\text{avg},\\
\end{align}

In [ ]:
from scipy.stats import ttest_rel

test_name = "t-test"
test_type = "paired"

# Tady byla chybka pri prepisovani puvodnich dat
tests_results[test_name][test_type] = {
            "two-sided": {},
            "less": {},
            "greater": {},
}




average_year_with_days = df_average_year[["mean","day_of_year"]]
print(len(average_year_with_days))
# prumerny rok obsahuje data i z 29. unora, coz je pouze pro prestupni roky

for year, data in df_annual.items():
    # Musim mergenout podle datumu!
    actual_year_df = pd.DataFrame({
        "day_of_year": data["date"].dt.day_of_year,
        "temperature": data["temperature_mean"]
    })

    # data nejdrive mergnu, abych zajistil ze se v kazdem roce porovnavaji spravne dny
    merged = pd.merge(
        actual_year_df,
        average_year_with_days,
        on="day_of_year",
        how="left",
    )

    actual_year_data = merged["temperature"]
    average_year_data = merged["mean"]


    p_value_two_sided = ttest_rel(actual_year_data, average_year_data, alternative="two-sided").pvalue
    p_value_less = ttest_rel(actual_year_data, average_year_data, alternative="less").pvalue
    p_value_greater = ttest_rel(actual_year_data, average_year_data, alternative="greater").pvalue

    tests_results[test_name][test_type]["two-sided"][year] = p_value_two_sided
    tests_results[test_name][test_type]["less"][year] = p_value_less
    tests_results[test_name][test_type]["greater"][year] = p_value_greater

Vykreslení průběhu p-hodnot pro jednotlivé roky.

In [ ]:
results = tests_results["t-test"]["paired"]
plot_test_results(results, title="Párový t-test: průběh p-hodnot v jednotlivých letech")

### Párový t-test je statisticky "silnější", protože rozptyl dat, se kterými počítáme, je mnohem menší

### 6. Analýza histogramu odchylek

Následující kód vykreslí histogram odchylek a zakreslí průběh normálního rozdělení se střední hodnotou a rozptylem odpovídající střední hodnotě a rozptylu odchylek.

In [ ]:
# Predpoklady ttestu
    # histogram odchylek by mel odpovida normalnimu rozdeleni
    # nezavislost pozorovani

df

In [ ]:
df_average_year

In [ ]:
merged_df_for_deviation = pd.merge(
    df,
    df_average_year,
    on="day_of_year",
    how="left"
   )
merged_df_for_deviation
# Pozor! std asi neni to co chci?

In [ ]:
deviation = merged_df_for_deviation["temperature_mean"] - merged_df_for_deviation["mean"]
print(deviation)

In [ ]:
import numpy as np
from scipy.stats import norm

mean_dev = np.mean(deviation)
std_dev = np.std(deviation)
x = np.linspace(deviation.min()-1, deviation.max()+1, 200)
pdf = norm.pdf(x, mean_dev, std_dev)    

plt.figure()
plt.hist(deviation, bins=50, density=True, alpha=0.6, label="Odchylky")
plt.plot(x, pdf, "r-", lw=2, label=f"N({mean_dev:.2f}, {std_dev:.2f}²)")
plt.title("Histogram denních teplotních odchylek a proložené normální rozdělení")
plt.xlabel("Odchylka teploty od průměrného dne [°C]")
plt.ylabel("Hustota pravděpodobnosti")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



Porovnáním histogramu odchylek a odpovídajícího normálního rozdělení vidíme, že tvar Gaussovy křivky věrně kopíruje rozdělení odchylek. Je vidět mírné levostranné zašikmení.

K T-testu je ještě nutné podotknout, že předpokládá nezávislost jednotlivých pozorování. To u denních teplot ovšem není zaručeno. Teploty po sobě jdoucích dnech jsou korelované.

Přestože histogram odchylek odpovídá normálnímu rozdělení (a tím se splňuje jeden předpoklad T-testu), druhý předpoklad není zcela splněn, což může vést k nadhodnocení statistické významnosti. (Tz. test je moc sebevědomý, příliš snadno a často zamítá $H_0$ i když by neměl).

### 7. Použití neparametrického párového testu

**Wilcoxonův test:**
- ptá se: "Jsou kladné a záporné odchylky rozložené symetricky kolem nuly?" 
- tj. je medián rozdílů == 0


Obecná formulace hypotéz:

* H0: odchylka od průměru je způsobena náhodnými poruchami.
* H1: odchylka od průměru je způsobena tím, že datová sada má jiný/nižší/vyšší průměr.

Konkrétní formulace při použití vybraného neparametrického testu:

\begin{align}
H_0:&\ \text{median}(d)=0,\\
H_1 (\text{rok je jiný}):& \text{median}(d)\ne 0,\\
H_1 (\text{rok je chladnější}):&\text{median}(d)<0 ,\\
H_1(\text{rok je teplejší}):&\text{median}(d)>0 ,\\
\end{align}

In [ ]:
from scipy.stats import wilcoxon

test_name = "wilcoxon"
test_type = "paired"

tests_results[test_name] = {
    test_type: {
        "two-sided": {},
        "less": {},
        "greater": {}
        }
}


average_year_with_days = df_average_year[["mean","day_of_year"]]
print(len(average_year_with_days))
# prumerny rok obsahuje data i z 29. unora, coz je pouze pro prestupni roky

for year, data in df_annual.items():
    # Musim mergenout podle datumu!
    actual_year_df = pd.DataFrame({
        "day_of_year": data["date"].dt.day_of_year,
        "temperature": data["temperature_mean"]
    })

    # data nejdrive mergnu, abych zajistil ze se v kazdem roce porovnavaji spravne dny
    merged = pd.merge(
        actual_year_df,
        average_year_with_days,
        on="day_of_year",
        how="left",
    )

    actual_year_data = merged["temperature"]
    average_year_data = merged["mean"]


    p_value_two_sided = wilcoxon(actual_year_data, average_year_data, alternative="two-sided").pvalue
    p_value_less = wilcoxon(actual_year_data, average_year_data, alternative="less").pvalue
    p_value_greater = wilcoxon(actual_year_data, average_year_data, alternative="greater").pvalue

    tests_results[test_name][test_type]["two-sided"][year] = p_value_two_sided
    tests_results[test_name][test_type]["less"][year] = p_value_less
    tests_results[test_name][test_type]["greater"][year] = p_value_greater

Vykreslení průběhu p-hodnot pro jednotlivé roky.

In [ ]:
plot_test_results(tests_results[test_name][test_type], title=f"Wilcoxonovů test: průběh p-hodnot v jednotlivých letech")

### 8. Analýza rozdílů použitých testů

In [ ]:
def get_sig_years(test_name, test_type, alternative, alpha=0.01):
    p_vals = tests_results[test_name][test_type][alternative]
    return {year for year, p in p_vals.items() if p < alpha}

def comparison_table(alpha = 0.01):
    alternatives = {
        "two-sided": "Jiný (two-sided)",
        "less": "Chladnější (less)",
        "greater": "Teplejší (greater)",
    }
    rows = []
    for alt_key, alt_label in alternatives.items():
        ind_t = get_sig_years("t-test", "independent", alt_key, alpha)
        rel_t = get_sig_years("t-test", "paired", alt_key, alpha)
        wilc = get_sig_years("wilcoxon", "paired", alt_key, alpha)

        rows.append({
            "Hypotéza": alt_label,
            "Nepárový t-test (počet)": len(ind_t),
            "Párový t-test (počet)": len(rel_t),
            "Pouze párový t-test oproti nepárovému [počet]": len(rel_t - ind_t),
            "|" : "|",
            "Wilcoxon párový (počet)": len(wilc),
            "Párový t-test oproti Wilcoxon [roky[": sorted(rel_t - wilc),
            "Wilcoxon oprorti Párový t-test [roky]": sorted(wilc - rel_t),
        })
    return rows

df_comparison = pd.DataFrame(comparison_table())
display(df_comparison)

Z analýzy vychází, že méně roků prochází nepárovým testem než párovým testem pro hladinu významnosti $\alpha = 0.01$.

Důvodem je robustnost párového testu vůči odchylkám. V nepárovém testu jsou odchylky hodnot napříč rokem obrovské. U párového testu nás zajímají odchylky pouze odpovídajících dnů a jsou tedy menší a "snadněji" se překonají.

---

Z analýzy vychází, že přibližně stejný počet roků prochází jak t-testem, tak neparametrickým testem pro hladinu významnosti $\alpha = 0.01$.

V tabulce výše jsou vidět konkrétní roky, ve kterých se parametrické testy liší oproti neparametrickému.

V teplotách se vyskytují extrémní mrazy (z histogramu odchylek jsou vidět odchylky až $-16^\circ\text{C}$), které nemají na kladné straně protějšek. T-test počítá s aritmetickým průměrem odchylek a tyto výrazné záporné odchylky strhly průměr směrem dolů.

Wilcoxonův test pracuje s pořadím. Výrazně mrazivý den nemá statisticky výraznější hodnotu než "obyčejný" chladný den.

Kvůli zápornému zešikmení leží medián (se kterým počítáme u neparametrického testu) o něco výše než jejich průměr.

T-test, který pracuje s průměrem tedy zachytil trochu více chladnějších let (o tři), zatímco Wilcoxonův neparametrický test zachytil více teplejších let (o jeden). 

Níže je tabulka srovnávající testy na hladině významnosti $\alpha=0.05$, jsou rozdíly mezi testy velice podobné jako pro hladinu významnosti $\alpha=0.01$.

In [ ]:
df_comparison = pd.DataFrame(comparison_table(alpha=0.05))
display(df_comparison)

### 9. Analýza trendu

**Co vlastně řešíme?**
- Trend bude nějak kladně nebo záporně strmý $\to$
- $\to$ budeme řešit jestli sklon naší přímky je dostatečně strmý na to aby šlo o skutečně dlouhodobý odhad

- (pod pojmem trend si tady představ lineární regresi)

**Pozor rok 2025 není úplný!**
- dojde k mírnému zkreslení výsledků
- (viz úkoly nad rámec zadání)

Obecná formulace hypotézy o trendu (slope):

* H0: trend je nulový
* H1: trend je nenulový.


Nalezení trendu, vykreslení a vyhodnocení p-hodnoty.

In [ ]:
np.array(years)
df_annual_stats["mean"]

In [ ]:
from scipy.stats import linregress

x_values = np.array(years)
y_values = df_annual_stats["mean"]

slope_lin_regress, intercept, r_value, p_value_lin_regress, std_err = linregress(x_values, y_values)



In [ ]:
def plot_lin_regress(
    x_values,
    y_values,
    slope_lin_regress,
    intercept,
    r_value,
    p_value_lin_regress,
    std_err,
    loc_name="Plzeň",
    ylabel="Průměrná roční teplota [°C]",
    title_prefix="Lineární trend průměrné roční teploty",
):
    alpha = 0.01
    print(f"=== {loc_name} ===")
    print(f"Sklon trendu (slope):      {slope_lin_regress:+.4f} °C / rok ({slope_lin_regress * 10:+.3f} °C / dekádu)")
    print(f"Úsek na ose y (intercept): {intercept:.2f} °C")
    print(f"Korelační koeficient (r):  {r_value:.4f} (R² = {r_value**2:.4f})")
    print(f"Směrodatná chyba sklonu:   {std_err:.4f} °C / rok")
    print(f"p-hodnota:                 {p_value_lin_regress:.2e}")

    if p_value_lin_regress < alpha:
        print(f"Závěr (p-value: {p_value_lin_regress:.2e} < alpha = {alpha}): Zamítáme H0. Lineární trend je statisticky významně odlišný od nuly.\n")
    else:
        print(f"Závěr (p-value: {p_value_lin_regress:.2e} > alpha = {alpha}): Nezamítáme H0. Nebyl prokázán statisticky významný trend.\n")

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(
        x_values, y_values,
        marker="o", markersize=4, linewidth=1.0, alpha=0.75,
        color="royalblue", label="Roční průměr"
    )
    ax.plot(
        x_values, intercept + slope_lin_regress * x_values,
        color="crimson", linewidth=2.2,
        label=f"Lineární trend: y = {slope_lin_regress:+.4f}·x {intercept:+.1f} ({slope_lin_regress * 10:+.2f} °C/dekádu)"
    )

    ax.set_title(f"{title_prefix} v {loc_name}")
    ax.set_xlabel("Rok")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

plot_lin_regress(
    x_values,
    y_values,
    slope_lin_regress,
    intercept,
    r_value,
    p_value_lin_regress,
    std_err,
    loc_name="Plzeň (1970-2025)",
    ylabel="Průměrná roční teplota [°C]",
    title_prefix="Lineární trend průměrné roční teploty",
)

Z grafu lineární regrese je patrný rostoucí trend $0.0406^\circ\text{C}$ za rok. 

Jednotlivé roky vykazují variabilitu teplot (např. výrazně chladný rok 2010), ale dlouhodobé zvýšení teploty za sledované období rozptyl dat kolem regresní přímky převyšuje.
Směrodatná odchylka odhadu sklonu ($0.0056$) je oproti samotnému sklonu ($0.0406$) velmi malá a výsledná p-hodnota je výrazně nižší (o několik řádů!) než hladina významnosti $\alpha = 0.01$.

Zamítáme proto nulovou hypotézu o nulovém trendu.

## Úkol 10 - Vypracování úkolu/ů nad rámec zadání
### Nalezené "problémy"
0. Neúplný rok 2025 zkresluje úkoly 4 a 9
- toto možná není nad rámec zadání a měl bych to ošetřit rovnou
- nezkresluje i jiné testy?
    - Párové testy nezkresluje, protože porovnáváme konkrétní dny 
1. Porovnávání roku se sebou samým (skoro)
- průměrný rok, který počítáme jako referenci pro testovaný rok, počítáme i z testovaného roku
- nejvíce ovlivňuje přestupní roky, ale je otázka jak velký to vůbec bude mít vliv
2. Předpoklad nezávislosti
- všechny použité testy předpokládají nezávislost pozorování
- to ovšem není u denních teplot zaručeno

## Úkol 10.0 - nekompletní rok 2025

In [ ]:
df_complete_years = df[df["year"] < 2025].copy()
df_annual_stats_complete = df_annual_stats[df_annual_stats.index < 2025].copy()

res_complete = linregress(df_annual_stats_complete.index.to_numpy(), df_annual_stats_complete["mean"].to_numpy())

print(f"Průměrná teplota roku 2025 (pouze 346 dnů): {df_annual_stats.loc[2025, 'mean']:.2f} °C")
print(f"Trend 1970-2025: slope = {slope_lin_regress:+.4f} °C/rok (p = {p_value_lin_regress:.2e})")
print(f"Trend 1970-2024: slope = {res_complete.slope:+.4f} °C/rok (p = {res_complete.pvalue:.2e})")


Pro t-test ořízneme průměrný rok na 346 dnů (pouze do 12.12.)

In [ ]:
days_2025 = df_annual[2025]["date"].dt.day_of_year
actual_2025 = df_annual[2025]["temperature_mean"]
avg_same_days = df_average_year[df_average_year["day_of_year"].isin(days_2025)]["mean"]

p_2025_unfair_two_sided = tests_results["t-test"]["independent"]["two-sided"][2025]
p_2025_unfair_greater = tests_results["t-test"]["independent"]["greater"][2025]
p_2025_fair_two_sided = ttest_ind(actual_2025, avg_same_days, equal_var=False, alternative="two-sided").pvalue
p_2025_fair_greater = ttest_ind(actual_2025, avg_same_days, equal_var=False, alternative="greater").pvalue

print(f"\nNepárový t-test pro rok 2025 (greater):")
print(f"Původní (346 dnů vs. 366 dnů): p = {p_2025_unfair_greater:.5f}")
print(f"Opravený (346 dnů vs. 346 dnů): p = {p_2025_fair_greater:.5f}")

print(f"\nNepárový t-test pro rok 2025 (two_sided):")
print(f"Původní (346 dnů vs. 366 dnů): p = {p_2025_unfair_two_sided:.5f}")
print(f"Opravený (346 dnů vs. 346 dnů): p = {p_2025_fair_two_sided:.5f}")

Výsledky původního nepárového t-testu:
```
Hladina významnosti alpha = 0.01
Významně odlišné roky (two-sided) [9]: [1980, 1987, 1996, 2014, 2015, 2018, 2019, 2023, 2024]
Významně chladnější roky (less)   [5]: [1978, 1980, 1985, 1987, 1996]
Významně teplejší roky (greater)  [8]: [2014, 2015, 2018, 2019, 2022, 2023, 2024, 2025]
```

Po přepočtení p-hodnot pouze pro 346 dnů v roce nám vychází výrazně vyšší p-hodnota. Výsledek pro oboustranný test zůstane stejný (rok stále není výrazně odlišný).

Ale pro jednostranný test (greater) p-hodnota překročila hladinu významnosti $\alpha = 0.01$ a nezamítáme tedy $H_0$.

## Úkol 10.1 - testovaný rok součástí referenčního průměru

In [ ]:
def get_leave_one_out_average_year(df_all, target_year):
    """
    Vrátí DataFrame se sloupci ['day_of_year', 'mean'] spočítaný
    ze všech let kromě target_year (prevence data leakage).
    """
    df_others = df_all[df_all["year"] != target_year]
    df_avg_loo = (
        df_others.groupby("day_of_year")["temperature_mean"]
        .mean()
        .reset_index()
    )
    return df_avg_loo

# Ukázka zapojení do existující smyčky (pro párový t-test i Wilcoxon):
loo_results = {
    "t-test": {"two-sided": {}, "less": {}, "greater": {}},
    "wilcoxon": {"two-sided": {}, "less": {}, "greater": {}},
}

for year, data in df_annual.items():
    actual_year_df = pd.DataFrame({
        "day_of_year": data["date"].dt.day_of_year,
        "temperature": data["temperature_mean"],
    })

    # Zde pouze nahradíme globální průměr za Leave-One-Out průměr
    avg_loo_df = get_leave_one_out_average_year(df, year)

    merged = pd.merge(actual_year_df, avg_loo_df, on="day_of_year", how="left")
    act = merged["temperature"]
    ref = merged["temperature_mean"]

    for alt in ("two-sided", "less", "greater"):
        loo_results["t-test"][alt][year] = ttest_rel(act, ref, alternative=alt).pvalue
        loo_results["wilcoxon"][alt][year] = wilcoxon(act, ref, alternative=alt).pvalue




In [ ]:
plot_test_results(loo_results["t-test"], title="Párový t-test: průběh p-hodnot v jednotlivých letech")

In [ ]:
plot_test_results(loo_results["wilcoxon"], title="Wilcoxonovů test: průběh p-hodnot v jednotlivých letech")

K žádnému rozdílu oproti předchozím výpočtům, kde se pracovalo s klasickým průměrem, nedošlo.

## Úkol 10.2 - autokorelace dat

In [ ]:
# 3A. Diagnostika: Výpočet a vykreslení autokorelace denních odchylek (lag 1 až 21 dnů)
max_lag = 21
lags = np.arange(1, max_lag + 1)
autocorrs = [deviation.autocorr(lag=k) for k in lags]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(lags, autocorrs, color="steelblue", edgecolor="black", alpha=0.8)
ax.axhline(0, color="black", linewidth=0.8)

# Orientační 95% pásmo nevýznamnosti pro bílý šum (±1.96 / sqrt(N))
conf_bound = 1.96 / np.sqrt(len(deviation))
ax.axhline(conf_bound, color="crimson", linestyle="--", label="95% hranice nezávislosti (bílý šum)")
ax.axhline(-conf_bound, color="crimson", linestyle="--")

ax.set_title("Autokorelační funkce (ACF) denních teplotních odchylek")
ax.set_xlabel("Zpoždění (lag) [dny]")
ax.set_ylabel("Autokorelační koeficient")
ax.set_xticks(lags)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

print(f"Autokorelace pro zpoždění 1 den:  r(1) = {autocorrs[0]:.3f}")
print(f"Autokorelace pro zpoždění 7 dnů:  r(7) = {autocorrs[6]:.3f}")

# 3B. Příprava dat pro nápravu: Agregace na měsíční průměry (12 pozorování za rok místo 365)
df["month"] = df["date"].dt.month
df_monthly = (
    df.groupby(["year", "month"])["temperature_mean"]
    .mean()
    .reset_index()
)

df_average_month = (
    df_monthly.groupby("month")["temperature_mean"]
    .mean()
    .rename("mean")
    .reset_index()
)

Místo toho abychom počítali s korelovanými daty dnů, budeme počítat s méně korelovanými daty měsíců. Snížíme tím ale počet pozorování z 365/366 na pouhých 12.

In [ ]:
monthly_results = {
    "t-test": {"two-sided": {}, "less": {}, "greater": {}},
    "wilcoxon": {"two-sided": {}, "less": {}, "greater": {}},
}

for year, group in df_monthly.groupby("year"):
    # Leave-One-Out průměr pro měsíce (ze všech let kromě testovaného roku)
    avg_month_loo = (
        df_monthly[df_monthly["year"] != year]
        .groupby("month")["temperature_mean"]
        .mean()
        .reset_index()
        .rename(columns={"temperature_mean": "mean"})
    )

    merged_m = pd.merge(group, avg_month_loo, on="month", how="left")
    act_m = merged_m["temperature_mean"]
    ref_m = merged_m["mean"]

    for alt in ("two-sided", "less", "greater"):
        monthly_results["t-test"][alt][year] = ttest_rel(act_m, ref_m, alternative=alt).pvalue
        monthly_results["wilcoxon"][alt][year] = wilcoxon(act_m, ref_m, alternative=alt).pvalue

In [ ]:
plot_test_results(monthly_results["t-test"], title="Párový t-test na měsíčních průměrech (odstranění denní autokorelace)")

Pro párový t-test se počet statisticky významných roků změnil z 30 na 8 pro obousměrnný test, počet chladnějších roků se změnil z 18 na 6 a počet teplejších roků se změnil z17 na 7.

In [ ]:
plot_test_results(monthly_results["wilcoxon"], title="Wilcoxonův test na měsíčních průměrech (odstranění denní autokorelace)")

Pro Wilcoxonův test se počet statisticky významných roků změnil z 31 na 6 pro obousměrnný test, počet chladnějších roků se změnil z 15 na 3 a počet teplejších roků se změnil z 18 na 6.

### Zkontrolování histogramu odchylek po zmenšení počtu pozorování (z denních na měsíční)

In [ ]:
merged_monthly_dev = pd.merge(
    df_monthly,
    df_average_month,
    on="month",
    how="left"
)
monthly_deviation = merged_monthly_dev["temperature_mean"] - merged_monthly_dev["mean"]

mean_m_dev = np.mean(monthly_deviation)
std_m_dev = np.std(monthly_deviation)
x_m = np.linspace(monthly_deviation.min() - 1, monthly_deviation.max() + 1, 200)
pdf_m = norm.pdf(x_m, mean_m_dev, std_m_dev)

plt.figure(figsize=(10, 5))
plt.hist(monthly_deviation, bins=25, density=True, alpha=0.6, label="Měsíční odchylky")
plt.plot(x_m, pdf_m, "r-", lw=2, label=f"N({mean_m_dev:.2f}, {std_m_dev:.2f}²)")
plt.title("Histogram měsíčních teplotních odchylek a proložené normální rozdělení")
plt.xlabel("Odchylka měsíční teploty od průměrného měsíce [°C]")
plt.ylabel("Hustota pravděpodobnosti")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Po agregaci dat na měsíční průměry došlo k výraznému poklesu počtu statisticky významných let na hladině $\alpha = 0.01$:
- párový t-test: počet významně odlišných let (two-sided) klesl z 30 na 8 (chladnějších z 18 na 6, teplejších ze 17 na 7).
- Wilcoxonův párový test: počet významně odlišných let (two-sided) klesl z 31 na 6 (chladnějších z 15 na 3, teplejších z 18 na 6).

Tento pokles je pravděpodobně způsoben dvěma faktory:
1. Odstranění denní autokorelace - to bylo cílem, protože 365 denních pozorování za rok nenese 365 nezávislých informací
2. Menší velikost vzorku ($n = 12$): Měsíční průměry jsou již téměř nezávislé, ale snížení počtu pozorování v roce z 365 na 12 snižuje statistickou sílu testu. 


**Přesto i při tomto přísnějším pohledu zůstávají nejextrémnější roky (např. chladné roky 1978, 1980, 1996 a nedávné velmi teplé roky 2014, 2015, 2019, 2020, 2022–2024) statisticky průkazné**.

# Úkol 11. Srovnání Plzně a města "Sarabuya" v Indonésii
- pozn.: nevím jak se město Sarabuya skloňuje

In [ ]:
# --------------------------------------------------
# Configuration
# --------------------------------------------------

location = {
    "name": "Surabaya",
    "continent": "Asia",
    "timezone": "Asia/Jakarta",
    "latitude": (-7, -14, -45),   # 7° 14' 45" S (jižní šířka -> záporné hodnoty)
    "longitude": (112, 44, 16),   # 112° 44' 16" E (východní délka)
}

for key in ("latitude", "longitude"):
    degrees, minutes, seconds = location[key]
    location[key] = degrees + minutes / 60 + seconds / 3600

start_date = "1970-01-01"
# end_date = "2025-12-31"
end_date = "2025-12-12" # zadani rika 12. prosince, ale tady bylo napsano 31. prosince
# Preklep, nebo kontrola toho ze jsem cetl zadani?
# nebo kontrola toho ze to byla hozeny cely do LLMka?
# who knows

# --------------------------------------------------
# Derived parameters
# --------------------------------------------------

# --------------------------------------------------
# Output
# --------------------------------------------------

print(f"Location: {location['name']}")
print(f"Continent: {location['continent']}")
print(f"Timezone: {location['timezone']}")
print(f"Latitude: {location['latitude']}")
print(f"Longitude: {location['longitude']}")
print(f"Start date: {start_date}")
print(f"End date: {end_date}")

In [ ]:
import datetime

date_start = datetime.date.fromisoformat(start_date)
date_end = datetime.date.fromisoformat(end_date)

n_days_expected = (date_end - date_start).days + 1

print(f"Start date: {date_start}")
print(f"End date: {date_end}")
print(f"Expected number of observations: {n_days_expected:,}")

In [ ]:
import requests
import pandas as pd
import json

# --------------------------------------------------
# Open-Meteo API
# --------------------------------------------------

url = "https://archive-api.open-meteo.com/v1/archive"

daily_variables = [
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
]

params = {
    "latitude": location["latitude"],
    "longitude": location["longitude"],
    "start_date": start_date,
    "end_date": end_date,
    "daily": daily_variables,
    "timezone": location["timezone"],
    "models": "era5",
}

# --------------------------------------------------
# Download data
# --------------------------------------------------

# Pro prevenci příliš častého volání api jsem po prvním spuštění data uložil do souboru data.json
# with requests.Session() as session:
#     response = session.get(url, params=params, timeout=(10, 120))
#     response.raise_for_status()
#     result = response.json()

# json_str = json.dumps(result, indent=2)
# with open("data_Surabaya.json", "w") as f:
#     f.write(json_str)


with open("data_Surabaya.json") as f:
    result = json.load(f)



# --------------------------------------------------
# Extract data
# --------------------------------------------------

if "daily" not in result:
    raise ValueError("Daily data are missing from the API response.")

daily = result["daily"]

df_surabaya = pd.DataFrame({
    "date": pd.to_datetime(daily["time"]),
    "temperature_mean": daily["temperature_2m_mean"],
    "temperature_min": daily["temperature_2m_min"],
    "temperature_max": daily["temperature_2m_max"],
})

# --------------------------------------------------
# Data validation
# --------------------------------------------------

print(f"Obtained observations: {len(df_surabaya):,}")
print(f"Expected observations: {n_days_expected:,}")

if len(df_surabaya) != n_days_expected:
    print("WARNING: Unexpected number of observations!")

if df_surabaya["date"].duplicated().any():
    raise ValueError("Duplicate dates detected.")

if not df_surabaya["date"].is_monotonic_increasing:
    raise ValueError("Dates are not in chronological order.")

missing = df_surabaya.isna().sum()

if missing.any():
    print("\nMissing values:")
    print(missing[missing > 0])

# --------------------------------------------------
# API metadata
# --------------------------------------------------

print("\nTemperature units:")
print(result.get("daily_units", {}))

print(f"\nTimezone: {result.get('timezone')}")
print(f"Elevation: {result.get('elevation')} m")

df.head()

In [ ]:
df_surabaya["year"] = df_surabaya["date"].dt.year
df_surabaya["day_of_year"] = df_surabaya["date"].dt.day_of_year

df_annual_surabaya = {
    year: group.drop(columns="year").reset_index(drop=True)
    for year, group in df_surabaya.groupby("year")
}

df_average_year_surabaya = (
    df_surabaya.groupby("day_of_year")["temperature_mean"]
    .agg(["mean", "std"])
    .reset_index()
)

df_average_year_surabaya

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    df_average_year["day_of_year"],
    df_average_year["mean"],
    color="royalblue",
    linewidth=2.2,
    label="Plzeň (mírný pás) - průměrný rok",
)
ax.fill_between(
    df_average_year["day_of_year"],
    df_average_year["mean"] - df_average_year["std"],
    df_average_year["mean"] + df_average_year["std"],
    color="royalblue",
    alpha=0.18,
    label="Plzeň ± 1 směrodatná odchylka",
)

ax.plot(
    df_average_year_surabaya["day_of_year"],
    df_average_year_surabaya["mean"],
    color="crimson",
    linewidth=2.2,
    label="Surabaya (tropy) - průměrný rok",
)
ax.fill_between(
    df_average_year_surabaya["day_of_year"],
    df_average_year_surabaya["mean"] - df_average_year_surabaya["std"],
    df_average_year_surabaya["mean"] + df_average_year_surabaya["std"],
    color="crimson",
    alpha=0.18,
    label="Surabaya ± 1 směrodatná odchylka",
)

ax.set_title("Srovnání průměrného ročního profilu teplot: Plzeň vs. Surabaya (1970–2025)")
ax.set_xlabel("Den v roce")
ax.set_ylabel("Průměrná denní teplota [°C]")
ax.set_xlim(1, 366)
ax.set_xticks(range(1, 367, 30))
ax.grid(True, alpha=0.3)
ax.legend(loc="center right")

fig.tight_layout()
plt.show()

Z dat je vidět, že průměrná teplota je v průběhu roku podstatně vyšší.


In [ ]:

df_annual_stats_surabaya = pd.DataFrame({
    year: {
        "mean": data["temperature_mean"].mean(),
    }
    for year, data in df_annual_surabaya.items()
    if len(data) >= 365  # automaticky vyřadí neúplný rok 2025 (346 dnů)
}).T
df_annual_stats_surabaya.index.name = "year"

x_values = df_annual_stats_surabaya.index.to_numpy()
y_values = df_annual_stats_surabaya["mean"]

slope_lin_regress, intercept, r_value, p_value_lin_regress, std_err = linregress(x_values, y_values)

plot_lin_regress(
    x_values,
    y_values,
    slope_lin_regress,
    intercept,
    r_value,
    p_value_lin_regress,
    std_err,
    loc_name="Surabaya",
    ylabel="Průměrná roční teplota [°C]",
    title_prefix="Lineární trend průměrné roční teploty",
)

Graf vypadá velice podobně, ale má jiné měřítko, které zkresluje jak moc průměrná teplota stoupá.

In [ ]:
def plot_multiple_lin_regress(series_dict, title="Srovnání lineárních trendů průměrné roční teploty", ylabel="Průměrná roční teplota [°C]"):
    fig, ax = plt.subplots(figsize=(14, 6))
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for idx, (label, (x_vals, y_vals)) in enumerate(series_dict.items()):
        x_arr = np.asarray(x_vals)
        y_arr = np.asarray(y_vals)
        color = colors[idx % len(colors)]

        res = linregress(x_arr, y_arr)

        ax.plot(
            x_arr,
            y_arr,
            marker="o",
            markersize=4,
            linewidth=1.0,
            alpha=0.55,
            color=color,
            label=f"{label} - data",
        )

        ax.plot(
            x_arr,
            res.intercept + res.slope * x_arr,
            linestyle="--",
            linewidth=2.3,
            color=color,
            label=f"{label} - trend ({res.slope * 10:+.2f} °C/dekádu, R²={res.rvalue**2:.2f})",
        )

    ax.set_title(title)
    ax.set_xlabel("Rok")
    ax.set_ylabel(f"{ylabel}")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2)

    fig.tight_layout()
    plt.show()

In [ ]:
plot_multiple_lin_regress(
    {
        "Plzeň (1970-2024)": (
            df_annual_stats_complete.index,
            df_annual_stats_complete["mean"],
        ),
        "Surabaya (1970-2024)": (
            df_annual_stats_surabaya.loc[:2024].index,
            df_annual_stats_surabaya.loc[:2024, "mean"],
        ),
    },
    title="Srovnání lineárního trendu průměrné roční teploty: Plzeň vs. Surabaya (1970–2024)",
)

V Sarabuye je stoupne teplota o $0.0245^\circ\text{C}$, což je méně než v Plzni, ale je stále k vidění rostoucí trend teplot.

V další buňce jsou zopakované úkoly 4, 5 a 7 (nepárový a párový t-test a Wilcoxonův neparametrický test).

In [ ]:
tests_results_Surabaya = {}

test_name = "t-test"
test_type = "independent"

tests_results_Surabaya[test_name] = {
        test_type: {
            "two-sided": {},
            "less": {},
            "greater": {},
        },
}

average_year = df_average_year_surabaya["mean"]

for year, data in df_annual_surabaya.items():

    actual_year = data["temperature_mean"] # vsechny data v roce

    p_value_two_sided = ttest_ind(actual_year,
                                   average_year,
                                   equal_var=False,
                                   alternative="two-sided").pvalue
    p_value_less = ttest_ind(actual_year,
                             average_year,
                             equal_var=False,
                             alternative="less").pvalue
    p_value_greater = ttest_ind(actual_year,
                                 average_year,
                                 equal_var=False,
                                 alternative="greater").pvalue  

    tests_results_Surabaya[test_name][test_type]["two-sided"][year] = p_value_two_sided
    tests_results_Surabaya[test_name][test_type]["less"][year] = p_value_less
    tests_results_Surabaya[test_name][test_type]["greater"][year] = p_value_greater



test_name = "t-test"
test_type = "paired"

tests_results_Surabaya[test_name][test_type] = {
            "two-sided": {},
            "less": {},
            "greater": {},
}

tests_results_Surabaya["wilcoxon"] = {
        test_type: {
            "two-sided": {},
            "less": {},
            "greater": {},
        },
}



average_year_with_days = df_average_year_surabaya[["mean","day_of_year"]]
print(len(average_year_with_days))
# prumerny rok obsahuje data i z 29. unora, coz je pouze pro prestupni roky

for year, data in df_annual_surabaya.items():
    # Musim mergenout podle datumu!
    actual_year_df = pd.DataFrame({
        "day_of_year": data["date"].dt.day_of_year,
        "temperature": data["temperature_mean"]
    })

    # data nejdrive mergnu, abych zajistil ze se v kazdem roce porovnavaji spravne dny
    merged = pd.merge(
        actual_year_df,
        average_year_with_days,
        on="day_of_year",
        how="left",
    )

    actual_year_data = merged["temperature"]
    average_year_data = merged["mean"]


    p_value_two_sided = ttest_rel(actual_year_data, average_year_data, alternative="two-sided").pvalue
    p_value_less = ttest_rel(actual_year_data, average_year_data, alternative="less").pvalue
    p_value_greater = ttest_rel(actual_year_data, average_year_data, alternative="greater").pvalue

    tests_results_Surabaya[test_name][test_type]["two-sided"][year] = p_value_two_sided
    tests_results_Surabaya[test_name][test_type]["less"][year] = p_value_less
    tests_results_Surabaya[test_name][test_type]["greater"][year] = p_value_greater

    p_value_two_sided_wilcoxon = wilcoxon(actual_year_data, average_year_data, alternative="two-sided").pvalue
    p_value_less_wilcoxon = wilcoxon(actual_year_data, average_year_data, alternative="less").pvalue
    p_value_greater_wilcoxon = wilcoxon(actual_year_data, average_year_data, alternative="greater").pvalue

    tests_results_Surabaya["wilcoxon"][test_type]["two-sided"][year] = p_value_two_sided_wilcoxon
    tests_results_Surabaya["wilcoxon"][test_type]["less"][year] = p_value_less_wilcoxon
    tests_results_Surabaya["wilcoxon"][test_type]["greater"][year] = p_value_greater_wilcoxon



In [ ]:

plot_test_results(tests_results_Surabaya["t-test"]["independent"], title=f"")
plot_test_results(tests_results_Surabaya["t-test"]["paired"], title=f"")
plot_test_results(tests_results_Surabaya["wilcoxon"]["paired"], title=f"")


Je k vidění, že nepárový test "netrpí" tolik na velké vnitro-roční rozptyly teplot a rozdíl je tedy mezi čtyřiceti třemi a čtyřiceti devíti lety (zatímco u Plzně to byl rozdíl mezi devíti a třiceti lety).

Celkově je statisticky odchýlených vždy alespoň čtyřicet tři let u všech oboustranných testů a dvacet u jednostranných testů.
Důvodem je menší rozptyl hodnot, protože denní teploty kolísají kolem průměrných hodnot minimálně. 

To způsobuje, že i relativně malý dlouhodobý posun stačí k překročení hladiny významnosti.

In [ ]:
def compare_locations(results_plzen, results_surabaya, alpha=0.01):
    tests = [
        ("Nepárový Welchův t-test", "t-test", "independent"),
        ("Párový t-test", "t-test", "paired"),
        ("Wilcoxonův párový test", "wilcoxon", "paired"),
    ]
    alternatives = {
        "two-sided": "Jiný (two-sided)",
        "less": "Chladnější (less)",
        "greater": "Teplejší (greater)",
    }

    rows = []
    for test_label, t_name, t_type in tests:
        for alt_key, alt_label in alternatives.items():
            p_plzen = results_plzen[t_name][t_type][alt_key]
            p_sur = results_surabaya[t_name][t_type][alt_key]

            sig_plzen = {y for y, p in p_plzen.items() if p < alpha}
            sig_sur = {y for y, p in p_sur.items() if p < alpha}
            n_years = len(p_plzen)

            rows.append({
                "Test": test_label,
                "Hypotéza": alt_label,
                "Plzeň [počet]": len(sig_plzen),
                "Surabaya [počet]": len(sig_sur),
                "Plzeň [%]": round(100 * len(sig_plzen) / n_years, 1),
                "Surabaya [%]": round(100 * len(sig_sur) / n_years, 1),
                "Společné roky [počet]": len(sig_plzen & sig_sur),
                "Pouze Surabaya [počet]": len(sig_sur - sig_plzen),
            })

    return pd.DataFrame(rows)

df_loc_comparison = compare_locations(tests_results, tests_results_Surabaya, alpha=0.01)
display(df_loc_comparison)

Z tabulky je vidět shoda u teplejších let. Jak u Wilcoxonova testu, tak u t-testu je vidět že velký počet let jsou společné jak pro Surabayu tak pro Plzeň. Pro Wilcoxonův test je to společných 16 z 18 pro Plzeň/z 24 pro Surabayu.  

# Úkol 12. Zkoumání extrémnosti počasí (testování hypotéz o rozptylu)
- cílem je zjistit zda se v Plzni v posleních letech mění pouze průměrná teplota nebo jestli se počasí stává extrémnějším (tedy jestli se zvyšuje rozptyl)
- k tomu využijeme [Leveneův test](https://en.wikipedia.org/wiki/Levene%27s_test)
- je to test rozptylu, který nepočítá s druhými mocninami odchylek, ale s absolutními hodnotami
- v podstatě převádí problém roztpylu na t-test pomocí absolutních odchylek od mediánu (odchylky od mediánu jsou tady vlastě nové náhodné veličiny) $Z_{ij} = |X_{ij} - \tilde X_i|$
    - původní data $X$ - průměr skupiny $\bar X$ je průměrná teplotní odchylka 
    - **po transformaci $Z_{ij}$ už není teplota, ale velikost rozptýlení daného dne $\to$ průměr těchto nových čísel $\bar Z$ je průměrná odchylka od mediánu!**
- data rozdělíme na dvě skupiny a odchylky od mediánu spočítáme pro obě skupiny
  - skupiny v našem budou dvě různá období (*1970-1989* / *2005-2024*)
- testová statistika porovnává rozdíl mezi skupinami oproti variabilitě uvnitř skupin [ANOVA](https://en.wikipedia.org/wiki/Analysis_of_variance) - dvouvýběrový t-test pro dvě skupiny
    - u Anovy chceme zjistit jestli dvě skupiny mají skutečně různý průměr nebo jestli rozdíl vznikl náhodou
    - pokud to chápu správně tak čitatel zlomku je rozdíl mezi skupinami
    - jmenovatel zlomku je variabilita uvnitř skupiny
    - pokud jsou body uvnitř skupiny blízko u sebe -> jmenovatel bude maličký a i malý rozdíl mezi skupinami dá velkou testovou statistiku W
    - naopak kdyz bude rozptyl uvnitř skupiny velký, jmenovatel vyjde velký a rozdíl mezi skupinami se "ztratí v šumu"
    - **POZOR! náš původní problém je stále analýza odchylek. ANOVA počitá s průměrem, proto jsme před samotným testováním odečítáním odchylek od mediánu**

- Naší nulovou hypotézou je, že se rozptyl hodnot mezi skupinami nezměnil
- Alternatvní hypotézou, je že se rozptyl statistiky změnil

- před testováním jsme museli nejdříve odstranit vliv ročního sezónního cyklu - mezi červnem a lednem víme že jsou velké rozdíly - nás zajímá rozptyl v rámci jednotlivých dnů oproti daného průměrného dne (tedy jak se odchýlila tepolota pro 6.6. oproti průměrnému 6.6.)
- s těmito rozptyly pak počítáme - z nich vezmeme medián v `levene(... center="median")` oproti kteremu zkoumáme odchylky 

In [ ]:
from scipy.stats import levene

def analyze_weather_extremes(
    df_location,
    location_name,
    early_period=(1970, 1989),
    recent_period=(2005, 2024),
    alpha=0.01,
):
    """
    Porovná rozptyl (extrémnost) denních teplotních odchylek mezi dvěma obdobími
    pomocí robustního Leveneova testu (center='median') a vykreslí histogramy.
    """
    data = df_location.copy()
    if "year" not in data.columns:
        data["year"] = data["date"].dt.year
    if "day_of_year" not in data.columns:
        data["day_of_year"] = data["date"].dt.day_of_year

    data = data[data["year"] < 2025].copy()
    avg_day = (
        data.groupby("day_of_year")["temperature_mean"]
        .mean()
        .rename("avg_temp")
        .reset_index()
    )
    data = pd.merge(data, avg_day, on="day_of_year", how="left")
    data["deviation"] = data["temperature_mean"] - data["avg_temp"]

    dev_early = data[data["year"].between(*early_period)]["deviation"]
    dev_recent = data[data["year"].between(*recent_period)]["deviation"]

    print(f"=== {location_name}: Analýza rozptylu denních teplotních odchylek ===")
    print(
        f"Období {early_period[0]}-{early_period[1]} (n = {len(dev_early)}): "
        f"mean = {dev_early.mean():+.2f} °C, std = {dev_early.std():.3f} °C, var = {dev_early.var():.3f} °C²"
    )
    print(
        f"Období {recent_period[0]}-{recent_period[1]} (n = {len(dev_recent)}): "
        f"mean = {dev_recent.mean():+.2f} °C, std = {dev_recent.std():.3f} °C, var = {dev_recent.var():.3f} °C²"
    )

    stat_levene, p_levene = levene(dev_early, dev_recent, center="median")
    print(f"Leveneův test: statistika W = {stat_levene:.4f}, p-hodnota = {p_levene:.4e} (~{p_levene:.3f})")

    if p_levene < alpha:
        print(f"Závěr (alpha = {alpha}): Zamítáme H0. Rozptyl (extrémnost) denních teplot se statisticky významně změnil.\n")
    else:
        print(f"Závěr (alpha = {alpha}): Nezamítáme H0. Nebyla prokázána statisticky významná změna rozptylu teplot.\n")

    x_min = min(dev_early.min(), dev_recent.min()) - 0.5
    x_max = max(dev_early.max(), dev_recent.max()) + 0.5
    bins = np.linspace(x_min, x_max, 50)

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(
        dev_early,
        bins=bins,
        density=True,
        alpha=0.5,
        color="royalblue",
        label=f"{early_period[0]}-{early_period[1]} (std = {dev_early.std():.2f} °C)",
    )
    ax.hist(
        dev_recent,
        bins=bins,
        density=True,
        alpha=0.5,
        color="crimson",
        label=f"{recent_period[0]}-{recent_period[1]} (std = {dev_recent.std():.2f} °C)",
    )

    ax.axvline(dev_early.mean(), color="royalblue", linestyle="--", linewidth=1.8)
    ax.axvline(dev_recent.mean(), color="crimson", linestyle="--", linewidth=1.8)

    ax.set_title(f"{location_name}: Porovnání rozdělení denních teplotních odchylek ({early_period[0]}-{early_period[1]} vs. {recent_period[0]}-{recent_period[1]})")
    ax.set_xlabel("Odchylka od průměrného dne v roce [°C]")
    ax.set_ylabel("Hustota pravděpodobnosti")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    plt.show()

    return {
        "location": location_name,
        "std_early": dev_early.std(),
        "std_recent": dev_recent.std(),
        "W": stat_levene,
        "p_value": p_levene,
    }


def plot_comparative_extremes(
    locations_dict,
    early_period=(1970, 1989),
    recent_period=(2005, 2024),
    alpha=0.01,
):
    """
    Vykreslí srovnání rozdělení denních teplotních odchylek pro libovolný počet lokalit
    do subplotů pod sebou se sdíleným měřítkem osy x i osy y (sharex=True, sharey=True).
    """
    n_locs = len(locations_dict)
    fig, axes = plt.subplots(
        n_locs, 1,
        figsize=(13, 4.5 * n_locs),
        sharex=True,
        sharey=True,
    )
    if n_locs == 1:
        axes = [axes]

    bins = np.arange(-20.0, 14.5, 0.5)

    for ax, (loc_name, df_loc) in zip(axes, locations_dict.items()):
        data = df_loc.copy()
        data["year"] = data["date"].dt.year
        data["day_of_year"] = data["date"].dt.day_of_year
        data = data[data["year"] < 2025].copy()

        avg_day = (
            data.groupby("day_of_year")["temperature_mean"]
            .mean()
            .rename("avg_temp")
            .reset_index()
        )
        data = pd.merge(data, avg_day, on="day_of_year", how="left")
        data["deviation"] = data["temperature_mean"] - data["avg_temp"]

        dev_early = data[data["year"].between(*early_period)]["deviation"]
        dev_recent = data[data["year"].between(*recent_period)]["deviation"]

        stat_w, p_val = levene(dev_early, dev_recent, center="median")

        ax.hist(
            dev_early,
            bins=bins,
            density=True,
            alpha=0.5,
            color="royalblue",
            label=f"{early_period[0]}–{early_period[1]} (mean = {dev_early.mean():+.2f} °C, std = {dev_early.std():.2f} °C)",
        )
        ax.hist(
            dev_recent,
            bins=bins,
            density=True,
            alpha=0.5,
            color="crimson",
            label=f"{recent_period[0]}–{recent_period[1]} (mean = {dev_recent.mean():+.2f} °C, std = {dev_recent.std():.2f} °C)",
        )

        ax.axvline(dev_early.mean(), color="royalblue", linestyle="--", linewidth=1.6)
        ax.axvline(dev_recent.mean(), color="crimson", linestyle="--", linewidth=1.6)

        ax.set_title(
            f"{loc_name}: Denní teplotní odchylky (Leveneův test: W = {stat_w:.2f}, p = {p_val:.2e})"
        )
        ax.set_ylabel("Hustota pravděpodobnosti")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper left")

    axes[-1].set_xlabel("Odchylka od průměrného dne v roce [°C]")
    fig.suptitle(
        f"Srovnání variability a posunu teplotních odchylek ({early_period[0]}–{early_period[1]} vs. {recent_period[0]}–{recent_period[1]})",
        fontsize=13,
        y=1.01,
    )
    fig.tight_layout()
    plt.show()

In [ ]:
res_extremes_plzen = analyze_weather_extremes(df, "Plzeň")
res_extremes_surabaya = analyze_weather_extremes(df_surabaya, "Surabaya")

In [ ]:
plot_comparative_extremes({
    "Plzeň (mírný pás)": df,
    "Surabaya (tropy)": df_surabaya,
})

Pro Plzeň vyšle p-hodnota $0.433$, což pro hladinu významnosti $\alpha =0.01$ nestačí k zamítnutí nulové hypotézy. V Plzni tedy (na první pohled) ke "zvětšení extrémnosti" nedošlo.

Naopak pro Surabayu, vyšla p-hodnota velice nízká (v řádu $10^{-5}$)a došlo k zamítnutí nulové hypotézy, ve prospěch alternativní, že rozptyl teplot se změnil.

Rozdíl mezi lokacemi je k vidění na posledním grafu, kde je vidět, že rozptyly teplot v Plzni byly i v dřívějším období dosti výrazné. Zatímco u roztpyly u Surabayi byly relativně malé.

V Plzni je ještě dobré zkoumat sezony zvlášť (letní měsíce a zimní měsíce). 

Je možné že se liší variabilita u výrazně chladných a výrazně teplých období.



In [ ]:
def analyze_seasonal_extremes(
    df_location,
    location_name,
    early_period=(1970, 1989),
    recent_period=(2005, 2024),
    alpha=0.01,
):
    """
    Otestuje změnu rozptylu denních odchylek zvlášť pro zimu (DJF) a léto (JJA).
    """
    data = df_location.copy()
    data["year"] = data["date"].dt.year
    data["month"] = data["date"].dt.month
    data["day_of_year"] = data["date"].dt.day_of_year
    data = data[data["year"] < 2025].copy()

    avg_day = data.groupby("day_of_year")["temperature_mean"].mean().rename("avg_temp").reset_index()
    data = pd.merge(data, avg_day, on="day_of_year", how="left")
    data["deviation"] = data["temperature_mean"] - data["avg_temp"]

    seasons = {
        "Zima (prosinec-únor)": [12, 1, 2],
        "Léto (červen-srpen)": [6, 7, 8],
    }

    print(f"=== {location_name}: Sezónní analýza rozptylu (Leveneův test) ===")
    for season_name, months in seasons.items():
        sub = data[data["month"].isin(months)]
        early = sub[sub["year"].between(*early_period)]["deviation"]
        recent = sub[sub["year"].between(*recent_period)]["deviation"]
        stat_w, p_val = levene(early, recent, center="median")
        print(
            f"{season_name:22s} | std {early_period[0]}–{early_period[1]}: {early.std():.3f} °C "
            f"-> {recent_period[0]}–{recent_period[1]}: {recent.std():.3f} °C | W = {stat_w:.3f}, p = {p_val:.4f}"
        )

analyze_seasonal_extremes(df, "Plzeň")

Je k vidění, že u zimního období zmenšení odchylky (klesla z $4.656^\circ\text{C}$ na $4.249^\circ\text{C}$), zatímco u léta došlo ke zvětšení odchylky (z $3.148^\circ\text{C}$ na $3.306^\circ\text{C}$).


U testu tedy došlo k tomu, že u jednoho období variabilita stoupla a u druhého období variabilita klesla (protože se počítá rozdíl rozptylů od mediánu rozptylů za celé dvacetileté období).

Toto jsou vlastně dva protichudné jevy, který Leveneův test neodhalí a naopak se v testu navzájem zkompenzují. Celkový roční rozptyl zůstal zdánlivě stejný.

P-hodnoty u jednotlivých období stále neklesnou pod hladinu významnosti $0.01$, ale je vidět, že je skok oproti p-hodnotě celoročnímu testu je výrazný (u celoročního testu $p=0.433$, u jednotlivých období $p=0.061$ a $p=0.053$), hodnoty klesly 7-krát!.

Z původní p-hodnoty $0.433$, by si tedy člověk mohl myslet, že se rozptylem hodnot napříč celým rokem nedělo vůbec nic, ale ve skutečnosti tomu není tak, po bližším zkoumání jednotlivých období! (Ale ne dost na to aby se přijala alternativní hypotéza stále)

## Úkol 13 - Analýza denní teplotní amplitudy
- v klimatologii se označuje jako DTR = Diurnal Temperature Range
- [Diural Temperature variation](https://en.wikipedia.org/wiki/Diurnal_temperature_variation) 
- je to rozdíl mezi nejvyšší a nejnižší teplotou v rámci jednoho dne
    - $DTR = T_{max} - T_{min}$
- v předchozích ukolech jsme pracovali s průměrnou denní/měsíční teplotou
- nyní budeme pracovat s maximálními a minimálními hodnotami 
- minimální hodnoty budou převážně v noci a maximální hodnoty kolem poledne 

- očekáváme, že v Plzni se vlivem jasných dnů vzduch rychle ohřeje a v noci zase rychle ochladí (mírný podnebný pás + žádný oceán)
- v Surabayi bude oceán fungovat jako teplotní setrvačník

- stejně jako u průměru vyhodnotíme pro všechny kompletní roky průměrnou roční amplitudu a zpracujeme pomocí již existující funkce pro lineární regresi
    - nulová hypotéza opět bude, že trend není statisticky významný


In [ ]:
def compute_annual_dtr(df_loc):
    """Spočítá průměrnou roční denní teplotní amplitudu (T_max - T_min) pro kompletní roky."""
    data = df_loc.copy()
    data["year"] = data["date"].dt.year
    data = data[data["year"] < 2025].copy()
    data["dtr"] = data["temperature_max"] - data["temperature_min"]
    return data.groupby("year")["dtr"].mean()

dtr_plzen = compute_annual_dtr(df)
dtr_surabaya = compute_annual_dtr(df_surabaya)


In [ ]:

for loc_name, dtr_series in [("Plzeň", dtr_plzen), ("Surabaya", dtr_surabaya)]:
    x_vals = dtr_series.index.to_numpy()
    y_vals = dtr_series.to_numpy()

    slope, intercept, r_val, p_val, std_err = linregress(x_vals, y_vals)
    plot_lin_regress(
        x_vals,
        y_vals,
        slope,
        intercept,
        r_val,
        p_val,
        std_err,
        loc_name=loc_name,
        ylabel="Průměrná denní amplituda T_max - T_min [°C]",
        title_prefix="Lineární trend denní teplotní amplitudy",
    )

In [ ]:
plot_multiple_lin_regress(
    {
        "Plzeň - denní amplituda (1970-2024)": (dtr_plzen.index, dtr_plzen.values),
        "Surabaya - denní amplituda (1970-2024)": (dtr_surabaya.index, dtr_surabaya.values),
    },
    title="Vývoj průměrné denní teplotní amplitudy (T_max - T_min)",
    ylabel="Denní teplotní amplituda - DTR"
)

Z hodnot je vidět, že v Plzni denní maxima rostou napříč roky o něco více, než noční minima.

Naopak pro Surabayu, noční minima se oteplují rychleji než denní maxima. (Trend jde dolů, tz. postupně se snižuje rozdíl $T_{max} - T_{min} \to$ $T_{min}$ roste rychleji než $T_{max}$).

Pro Plzeň trend DTR není statisticky odlišný od nuly, ale pro Surabayu ano. Tam je trend záporný a dostatečně statisticky významný k zamítnutí nulové hypotézy, ve prospěch alternativní.


## Závěr
V semestrální práci jsme provedli statistickou analýzu pro Plzeň pomocí párového a nepárového t-testu, neparametrického Wilcoxonova párového testu a nakonec i pomocí zkoumání trendu lineární regrese. 

V rámci dodatečných úkolů poté byly opraveny nedokonalosti ze zadání (práce s nekompletním rokem 2025, práce s autokorelovanými daty a vynechání testovaného roku z průměru), byla provedena statistická analýza pro město Surabaya v Indonésii a nakonec byla vyhodnocena analýza rozptylu a analýza denní teplotní amplitudy u obou měst.

V rámci prvních devíti úkolů bylo zjištěno, že napříč roky v Plzni došlo k oteplení (nárůst průměrné roční teploty o $0{,}041\ ^\circ\text{C}/\text{rok}$). U t-testů i u neparametrického testu bylo zjištěno, že velká část roků má statisticky významný rozdíl střední hodnoty. Dřívější roky bývaly častěji chladnější než průměr a naopak poslední roky jsou teplejší než průměr. 

Je nutné poznamenat, že rozptyl hodnot teplot v Plzni napříč roky byl vždy výrazný a v rámci úkolu 12 bylo zjištěno, že nedošlo ke statisticky významné změně tohoto celoročního rozptylu. Původní testy také předpokládají nezávislost pozorování, což u denních teplot není skutečností.

Při přechodu na měsíční teploty v rámci úkolu 10 počet statisticky významných let klesl (z desítek na jednotky). Je ale stále k vidění, že v posledních letech došlo ke statisticky významnému oteplení (jak u Wilcoxonova testu, tak u parametrického párového t-testu).

V rámci města Surabaya bylo nalezeno podstatně více statisticky významných let, hlavně díky menšímu rozptylu teplot (teploty se udržují více kolem průměru $27\ ^\circ\text{C}$). 

Při porovnávání měst bylo zjištěno, že pro Surabaju se na rozdíl od Plzně statisticky významně změnil rozptyl hodnot a také došlo k mírnému oteplení při analýze trendu. U porovnávání denních teplotních amplitud nebyl u Plzně zjištěn žádný statisticky významný trend, zatímco u Surabaje dochází k rychlejšímu oteplování minimálních hodnot (teplot v noci) oproti denním maximům.

Všechny úkoly zadání (i dobrovolné) byly splněny a práci považuji za dokončenou.